# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\dev\llm-data-analysis-course\notebooks
프로젝트 루트: c:\dev\llm-data-analysis-course
전처리 데이터 폴더: c:\dev\llm-data-analysis-course\data\processed
보고서 폴더: c:\dev\llm-data-analysis-course\reports


## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


전처리 데이터 불러오기 완료


## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [3]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (764, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 0
주문 시작일: 2025-09-14 00:00:00
주문 종료일: 2026-09-13 00:00:00


## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [5]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions


,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [6]:
customers['age'].describe()


count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64

In [7]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,성남,21,14.00
1,광주,17,11.33
2,부산,16,10.67
3,대구,15,10.00
4,서울,15,10.00
5,울산,14,9.33
6,인천,14,9.33
7,대전,14,9.33
8,수원,13,8.67
9,고양,11,7.33


In [8]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,F,84,56.0
1,M,66,44.0


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [9]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,스포츠,19,19.0
1,전자기기,17,17.0
2,생활용품,16,16.0
3,뷰티,16,16.0
4,도서,14,14.0
5,패션,11,11.0
6,식품,7,7.0


In [10]:
products['price'].describe()


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

In [11]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
4,식품,7,137143.0,67000,197000
1,뷰티,16,117688.0,5000,200000
6,패션,11,115909.0,28000,198000
3,스포츠,19,111579.0,10000,196000
0,도서,14,106857.0,25000,174000
5,전자기기,17,101588.0,5000,165000
2,생활용품,16,96438.0,11000,188000


## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [12]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,184,61.33
1,cancelled,64,21.33
2,refunded,52,17.33


In [13]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,79,26.33
1,naver_pay,77,25.67
2,bank_transfer,74,24.67
3,card,70,23.33


In [14]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-09-14 00:00:00
주문 종료일: 2026-09-13 00:00:00
주문 월 개수: 13


## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [15]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [16]:
sales_items = order_items.merge(
    products,
    on='product_id',
    how='left',
)

print('병합 전 order_items:', order_items.shape)
print('병합 후 sales_items:', sales_items.shape)
print('상품명 누락:', sales_items['product_name'].isna().sum())
print('카테고리 누락:', sales_items['category'].isna().sum())

sales_items.head()


병합 전 order_items: (764, 6)
병합 후 sales_items: (764, 9)
상품명 누락: 0
카테고리 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price
0,1,1,100,3,102000,306000,도서 상품 100,도서,102000
1,2,1,87,5,25000,125000,도서 상품 087,도서,25000
2,3,1,7,3,142000,426000,도서 상품 007,도서,142000
3,4,1,9,3,193000,579000,스포츠 상품 009,스포츠,193000
4,5,2,72,4,189000,756000,뷰티 상품 072,뷰티,189000


## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [17]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
3,스포츠,468,50174000,19.63
1,뷰티,376,47551000,18.60
5,전자기기,401,41003000,16.04
2,생활용품,390,34839000,13.63
4,식품,240,33597000,13.14
0,도서,238,24645000,9.64
6,패션,220,23801000,9.31


In [18]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,스포츠,468,50174000,19.63,111579.0
1,뷰티,376,47551000,18.60,117688.0
2,전자기기,401,41003000,16.04,101588.0
3,생활용품,390,34839000,13.63,96438.0
4,식품,240,33597000,13.14,137143.0
5,도서,238,24645000,9.64,106857.0
6,패션,220,23801000,9.31,115909.0


해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [19]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left',
)

order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

print('병합 후 order_sales:', order_sales.shape)
print('order_date 누락:', order_sales['order_date'].isna().sum())

order_sales.head()


병합 후 order_sales: (764, 12)
order_date 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,100,3,102000,306000,123,2026-07-13,card,completed,2026-07,Monday
1,2,1,87,5,25000,125000,123,2026-07-13,card,completed,2026-07,Monday
2,3,1,7,3,142000,426000,123,2026-07-13,card,completed,2026-07,Monday
3,4,1,9,3,193000,579000,123,2026-07-13,card,completed,2026-07,Monday
4,5,2,72,4,189000,756000,77,2025-09-28,naver_pay,cancelled,2025-09,Sunday


In [20]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-09,5640000,10,564000.0
1,2025-10,25451000,27,942630.0
2,2025-11,17912000,21,852952.0
3,2025-12,32113000,35,917514.0
4,2026-01,28202000,29,972483.0
5,2026-02,13990000,16,874375.0
6,2026-03,24367000,34,716676.0
7,2026-04,24446000,28,873071.0
8,2026-05,24775000,27,917593.0
9,2026-06,19882000,26,764692.0


월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


In [31]:
# STEP 3~4. completed 범위 집계와 총합 검증
completed_items = order_sales[order_sales['order_status'] == 'completed']
rows_before_merge = len(completed_items)

completed_items = completed_items.merge(
    products[['product_id', 'category']],
    on='product_id',
    how='left',
)

completed_total = completed_items['line_total'].sum()

completed_category_sales = (
    completed_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)
completed_category_sales['sales_ratio'] = (
    completed_category_sales['total_sales'] / completed_category_sales['total_sales'].sum() * 100
).round(2)

completed_monthly_sales = (
    completed_items
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)
completed_monthly_sales['avg_order_value'] = (
    completed_monthly_sales['total_sales'] / completed_monthly_sales['order_count']
).round(0)

category_total = completed_category_sales['total_sales'].sum()
monthly_total = completed_monthly_sales['total_sales'].sum()

print('completed 주문 상세 행 수:', len(completed_items), '/ 병합 전:', rows_before_merge)
print('카테고리 누락:', completed_items['category'].isna().sum())
print('completed 주문 수(주문 상세가 있는 주문):', completed_items['order_id'].nunique(),
      '/ orders의 completed 주문 수:', (orders['order_status'] == 'completed').sum())
print()
print('completed 원본 합계:', f'{completed_total:,}')
print('카테고리별 합계   :', f'{category_total:,}')
print('월별 합계         :', f'{monthly_total:,}')
print('원본 = 카테고리 합계:', completed_total == category_total)
print('원본 = 월별 합계    :', completed_total == monthly_total)
print()
print(completed_category_sales.to_string(index=False))
print(completed_monthly_sales.to_string(index=False))

completed 주문 상세 행 수: 474 / 병합 전: 474
카테고리 누락: 0
completed 주문 수(주문 상세가 있는 주문): 184 / orders의 completed 주문 수: 184

completed 원본 합계: 148,990,000
카테고리별 합계   : 148,990,000
월별 합계         : 148,990,000
원본 = 카테고리 합계: True
원본 = 월별 합계    : True

category  total_quantity  total_sales  sales_ratio
     스포츠             295     31743000        21.31
    전자기기             259     26400000        17.72
    생활용품             272     23915000        16.05
      뷰티             223     23383000        15.69
      식품             133     16573000        11.12
      도서             149     16389000        11.00
      패션             111     10587000         7.11
order_month  total_sales  order_count  avg_order_value
    2025-09      2650000            5         530000.0
    2025-10     16970000           19         893158.0
    2025-11     11021000           13         847769.0
    2025-12     22314000           24         929750.0
    2026-01     13303000           16         831438.0
    2026-02      5318000  

## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [21]:
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left',
)

group_columns = ['customer_id', 'city']
if 'name' in customer_sales_base.columns:
    group_columns = ['customer_id', 'name', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

customer_sales.head(10)


,customer_id,name,city,order_count,total_sales,avg_order_value
79,102,이진호,고양,5,5977000,1195400.0
93,117,김지원,성남,6,5552000,925333.0
67,83,전은경,수원,5,4951000,990200.0
118,142,하서연,서울,3,4862000,1620667.0
34,40,박예준,서울,5,4857000,971400.0
24,30,이민재,서울,7,4853000,693286.0
0,3,이경수,성남,3,4668000,1556000.0
10,14,윤성호,대전,3,4267000,1422333.0
122,146,김숙자,성남,3,4108000,1369333.0
87,111,류정훈,광주,4,4067000,1016750.0


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [22]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [23]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_category_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_customer_city.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_customer_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_eda_questions.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_monthly_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/reports/ch06_product_category.csv')]

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [24]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)


EDA 요약 보고서 저장 완료: c:\dev\llm-data-analysis-course\reports\ch06_eda_summary.md


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [25]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [26]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
3,스포츠,468,50174000,19.63
1,뷰티,376,47551000,18.60
5,전자기기,401,41003000,16.04
2,생활용품,390,34839000,13.63
4,식품,240,33597000,13.14


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [32]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.

completed_sales = (
    order_sales[order_sales['order_status'] == 'completed']
    .merge(products[['product_id', 'category']], on='product_id', how='left')
    .merge(customers[['customer_id', 'city']], on='customer_id', how='left')
)
completed_total = completed_sales['line_total'].sum()

print('completed_sales 행 수:', len(completed_sales))
print('카테고리 누락:', completed_sales['category'].isna().sum())
print('도시 누락:', completed_sales['city'].isna().sum())
print('completed 원본 합계:', f'{completed_total:,}')

city_customers = (
    customers
    .groupby('city', as_index=False)
    .agg(customer_count=('customer_id', 'nunique'))
)

city_sales = (
    completed_sales
    .groupby('city', as_index=False)
    .agg(
        buyer_count=('customer_id', 'nunique'),
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
)

city_summary = city_customers.merge(city_sales, on='city', how='left').fillna(0)
city_summary[['buyer_count', 'order_count', 'total_sales']] = (
    city_summary[['buyer_count', 'order_count', 'total_sales']].astype(int)
)
city_summary['sales_per_customer'] = (
    city_summary['total_sales'] / city_summary['customer_count']
).round(0)
city_summary['customer_ratio'] = (
    city_summary['customer_count'] / city_summary['customer_count'].sum() * 100
).round(2)
city_summary['sales_ratio'] = (
    city_summary['total_sales'] / city_summary['total_sales'].sum() * 100
).round(2)
city_summary = city_summary.sort_values('total_sales', ascending=False)

display(city_summary)
print('고객 수 합계:', city_summary['customer_count'].sum(), '/ 전체 고객:', len(customers))
print('도시별 합계 = 원본 completed 합계:', city_summary['total_sales'].sum() == completed_total)


completed_sales 행 수: 474
카테고리 누락: 0
도시 누락: 0
completed 원본 합계: 148,990,000


,city,customer_count,buyer_count,order_count,total_sales,sales_per_customer,customer_ratio,sales_ratio
6,성남,21,17,31,24099000,1147571.0,14.00,16.17
0,고양,11,9,18,19116000,1737818.0,7.33,12.83
5,서울,15,12,29,19097000,1273133.0,10.00,12.82
8,울산,14,10,20,15380000,1098571.0,9.33,10.32
1,광주,17,11,17,14264000,839059.0,11.33,9.57
4,부산,16,8,13,13604000,850250.0,10.67,9.13
9,인천,14,10,18,12832000,916571.0,9.33,8.61
3,대전,14,8,14,11792000,842286.0,9.33,7.91
7,수원,13,7,14,10620000,816923.0,8.67,7.13
2,대구,15,8,10,8186000,545733.0,10.00,5.49


고객 수 합계: 150 / 전체 고객: 150
도시별 합계 = 원본 completed 합계: True


In [33]:
# 과제 2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
category_compare = (
    products
    .groupby('category', as_index=False)
    .agg(product_count=('product_id', 'count'))
    .merge(
        completed_sales
        .groupby('category', as_index=False)
        .agg(total_quantity=('quantity', 'sum'), total_sales=('line_total', 'sum')),
        on='category',
        how='left',
    )
)
category_compare['sales_per_product'] = (
    category_compare['total_sales'] / category_compare['product_count']
).round(0)
category_compare['product_rank'] = (
    category_compare['product_count'].rank(ascending=False, method='min').astype(int)
)
category_compare['sales_rank'] = (
    category_compare['total_sales'].rank(ascending=False, method='min').astype(int)
)
category_compare = category_compare.sort_values('total_sales', ascending=False)

display(category_compare)
print(
    '상품 수와 completed 매출의 순위 상관(Spearman):',
    round(category_compare['product_count'].corr(category_compare['total_sales'], method='spearman'), 3),
)
print('카테고리 합계 = 원본 completed 합계:', category_compare['total_sales'].sum() == completed_total)

,category,product_count,total_quantity,total_sales,sales_per_product,product_rank,sales_rank
3,스포츠,19,295,31743000,1670684.0,1,1
5,전자기기,17,259,26400000,1552941.0,2,2
2,생활용품,16,272,23915000,1494688.0,3,3
1,뷰티,16,223,23383000,1461438.0,3,4
4,식품,7,133,16573000,2367571.0,7,5
0,도서,14,149,16389000,1170643.0,5,6
6,패션,11,111,10587000,962455.0,6,7


상품 수와 completed 매출의 순위 상관(Spearman): 0.883
카테고리 합계 = 원본 completed 합계: True


### 과제 1 해석
- 관찰: 성남이 고객 수(21명, 14.0%)와 completed 매출(24,099,000원, 16.17%) 모두 1위이다. 고양은 고객 수가 가장 적은데(11명, 7.33%) 매출은 2위(19,116,000원)이고 고객 1명당 매출(1,737,818원)이 가장 높다. 대구는 고객 수가 15명인데 매출이 최하위(8,186,000원, 5.49%)이고 고객당 매출(545,733원)이 가장 낮다. 광주는 고객 수 2위(17명)인데 매출은 5위이다. 구매 고객 비율(buyer_count / customer_count)은 고양 9/11, 성남 17/21, 서울 12/15가 약 80%대인 반면 대구 8/15, 부산 8/16은 50%대이다.
- 가설: 도시별 매출 차이는 구매 고객 비율과 구매 고객당 주문 수·금액의 차이에서 나올 수 있다. 다만 도시별 고객이 11~21명뿐이라 우연일 수도 있다.
- 추가 검증: 도시별 구매 고객 비율, 구매 고객당 주문 수, 평균 주문 금액으로 매출 차이를 나눠 본다.

### 과제 2 해석

**관찰:** 상품 수 상위 4개 카테고리(스포츠 19, 전자기기 17, 생활용품 16, 뷰티 16)가 completed 매출 상위 4개(스포츠, 전자기기, 생활용품, 뷰티)와 같다. 상품 수와 매출의 Spearman 순위 상관은 약 0.88이다. 예외는 식품으로, 상품 수는 7개로 가장 적은데 매출은 5위이고 상품 1개당 매출(약 2,367,571원)은 가장 높다.

**판단:** 상품 수가 많은 카테고리가 대체로 매출도 높지만, 식품처럼 상품 수가 적어도 매출이 높은 카테고리가 있어 상품 수만으로 매출을 설명할 수는 없다.

**한계:** 카테고리가 7개뿐이라 상관 값이 하나의 카테고리에 크게 흔들릴 수 있고, 상관은 인과를 뜻하지 않는다. 상품 수가 많아서 매출이 높은지, 인기 상품군에 상품이 많이 등록되었는지는 이 데이터로 알 수 없다. 수량과 단가 중 무엇의 영향인지는 추가 분석이 필요하다.

In [35]:
# 과제 3. 결제수단별 평균 주문 금액을 계산하세요.
# 평균 주문 금액은 주문 단위로 먼저 합산한 뒤 평균을 계산한다(주문 상세 행 평균과 구분).
completed_orders = (
    completed_sales
    .groupby(['order_id', 'payment_method'], as_index=False)
    .agg(order_total=('line_total', 'sum'))
)

payment_avg = (
    completed_orders
    .groupby('payment_method', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('order_total', 'sum'),
        avg_order_value=('order_total', 'mean'),
        median_order_value=('order_total', 'median'),
    )
    .sort_values('avg_order_value', ascending=False)
)
payment_avg['avg_order_value'] = payment_avg['avg_order_value'].round(0)
payment_avg['median_order_value'] = payment_avg['median_order_value'].round(0)

display(payment_avg)
print('주문 수 합계:', payment_avg['order_count'].sum())
print('결제수단별 합계 = 원본 completed 합계:', payment_avg['total_sales'].sum() == completed_total)

,payment_method,order_count,total_sales,avg_order_value,median_order_value
3,naver_pay,51,43500000,852941.0,860000.0
1,card,39,31806000,815538.0,837000.0
2,kakao_pay,49,39342000,802898.0,804000.0
0,bank_transfer,45,34342000,763156.0,713000.0


주문 수 합계: 184
결제수단별 합계 = 원본 completed 합계: True


### 과제 3 해석
- 관찰: 결제수단별 평균 주문 금액은 naver_pay 852,941원이 가장 높고 bank_transfer 763,156원이 가장 낮으며, 차이는 약 89,785원(낮은 쪽 대비 약 12%)이다. 중앙값의 순위(860,000 > 837,000 > 804,000 > 713,000)도 평균과 같다. 결제수단별 completed 주문 수는 39~51건이다.
- 가설: 차이가 크지 않고 표본이 작아 우연일 수 있다. 결제수단별로 구매하는 카테고리 구성이 달라서 생긴 차이일 수도 있다.
- 추가 검증: 결제수단별 주문 상태 비율(LLM 질문 5)과 주문 금액 분포를 확인한다.

In [36]:
# 과제 4. 월별·카테고리별 매출 요약표를 만들어 보세요.
monthly_category_sales = completed_sales.pivot_table(
    index='order_month',
    columns='category',
    values='line_total',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='합계',
)

category_order = (
    completed_sales
    .groupby('category')['line_total']
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)
monthly_category_sales = monthly_category_sales[category_order + ['합계']]

display(monthly_category_sales)
print('총합 = 원본 completed 합계:', monthly_category_sales.loc['합계', '합계'] == completed_total)

category,스포츠,전자기기,생활용품,뷰티,식품,도서,패션,합계
order_month,,,,,,,,
2025-09,269000,498000,893000,580000,0,410000,0,2650000
2025-10,2961000,4377000,1162000,3204000,2481000,774000,2011000,16970000
2025-11,3610000,2909000,1610000,1130000,422000,844000,496000,11021000
2025-12,4848000,3158000,5866000,4397000,1283000,1831000,931000,22314000
2026-01,2582000,3232000,1243000,2082000,1887000,1589000,688000,13303000
2026-02,695000,916000,439000,345000,1933000,990000,0,5318000
2026-03,4978000,3282000,2138000,1996000,1895000,2549000,691000,17529000
2026-04,2452000,1230000,1193000,2310000,1457000,1782000,1408000,11832000
2026-05,5368000,1161000,2446000,2330000,2094000,1040000,1906000,16345000


총합 = 원본 completed 합계: True


### 과제 4 해석
- 관찰: 2025-12(22,314,000원)는 2025-11(11,021,000원)보다 11,293,000원 늘었고, 그중 생활용품(+4,256,000원)과 뷰티(+3,267,000원)의 증가가 컸다. 가장 큰 단일 셀은 2025-12 생활용품(5,866,000원, 해당 월 합계의 약 26.3%)이다. 0원인 셀도 있다(2025-09 식품/패션, 2026-02 패션, 2026-08 식품/도서, 2026-09 뷰티/패션).
- 가설: 월x카테고리 셀 하나가 주문 상세 몇 건에 불과해 값이 크게 흔들릴 수 있고, 0원은 판매가 없었거나 우연일 수 있다. 2025-12 상승은 몇몇 카테고리가 함께 올랐기 때문일 수 있다.
- 추가 검증: 월별 변화를 카테고리별 주문 수와 평균 단가로 나눠 본다(5단계 우선순위 3과 연결).

In [39]:
# 과제 5. 고객별 order_count와 avg_order_value를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 만든다.
customer_completed = (
    completed_sales
    .groupby('customer_id', as_index=False)
    .agg(order_count=('order_id', 'nunique'), total_sales=('line_total', 'sum'))
)
customer_completed['avg_order_value'] = (
    customer_completed['total_sales'] / customer_completed['order_count']
).round(0)

display(customer_completed[['order_count', 'total_sales', 'avg_order_value']].describe())
display(
    customer_completed['order_count']
    .value_counts()
    .sort_index()
    .rename_axis('order_count')
    .reset_index(name='customer_count')
)

repeat_min_orders = 2
high_value_threshold = customer_completed['avg_order_value'].quantile(0.75)
print('반복 구매 기준: order_count >=', repeat_min_orders)
print('고액 기준(구매 고객 평균 주문 금액의 75번째 백분위수):', f'{high_value_threshold:,.0f}')

def classify_customer(row):
    if row['order_count'] >= repeat_min_orders:
        return '반복 구매'
    if row['avg_order_value'] >= high_value_threshold:
        return '고액 단발'
    return '일반 단발'

customer_completed['customer_type'] = customer_completed.apply(classify_customer, axis=1)

customer_type_summary = (
    customer_completed
    .groupby('customer_type', as_index=False)
    .agg(
        customer_count=('customer_id', 'count'),
        total_sales=('total_sales', 'sum'),
        avg_order_value=('avg_order_value', 'mean'),
    )
    .assign(avg_order_value=lambda df: df['avg_order_value'].round(0))
)

display(customer_type_summary)
print('completed 구매 고객 수:', len(customer_completed))
print('completed 구매 이력이 없는 고객 수:', len(customers) - len(customer_completed))

,order_count,total_sales,avg_order_value
count,100.00000,1.000000e+02,1.000000e+02
mean,1.84000,1.489900e+06,8.250658e+05
std,0.96106,9.585402e+05,3.973710e+05
min,1.00000,1.120000e+05,5.600000e+04
25%,1.00000,8.287500e+05,5.282500e+05
50%,2.00000,1.296500e+06,8.070000e+05
75%,2.00000,2.008500e+06,1.061750e+06
max,5.00000,4.100000e+06,1.931000e+06


,order_count,customer_count
0,1,44
1,2,37
2,3,12
3,4,5
4,5,2


반복 구매 기준: order_count >= 2
고액 기준(구매 고객 평균 주문 금액의 75번째 백분위수): 1,061,750


,customer_type,customer_count,total_sales,avg_order_value
0,고액 단발,15,20315000,1354333.0
1,반복 구매,56,111479000,803493.0
2,일반 단발,29,17196000,592966.0


completed 구매 고객 수: 100
completed 구매 이력이 없는 고객 수: 50


### 과제 5. 반복 구매 고객 vs 고액 단발 구매 고객 구분 기준

노트북  셀 14 설명처럼, 총 구매 금액이 같아도 한 번 고액으로 산 고객과 여러 번 반복해 산 고객은 다르게 해석해야 한다. 그래서 `total_sales` 하나가 아니라 `order_count`와 `avg_order_value`를 함께 본다.

| 구분 | 기준 | 이유 |
|---|---|---|
| 반복 구매 | completed 주문 2건 이상 | 구매가 한 번 이상 반복됨 |
| 고액 단발 | completed 주문 1건이면서 평균 주문 금액이 구매 고객 상위 25%(75번째 백분위수 이상) | 한 번의 주문이 큰 고객 |
| 일반 단발 | 위 두 조건에 해당하지 않는 completed 주문 1건 고객 | 나머지 |
| (별도) 구매 이력 없음 | completed 주문 0건 | 위 분류에서 제외하고 건수만 따로 확인 |

- 결과:
1. 2번 이상 구매한 고객을 반복 구매 고객으로 설정하였다.
2. 고액 단발 고객은 고객의 평균 주문 금액을 작은 순서로 줄 세웠을 때 75% 지점의 값을 threshold으로 설정하여, 소비 금액이 이 값보다 큰 소비자의 수를 집계하였다.
3. completed 구매 고객 100명 중 반복 구매는 56명, 고액 단발은 15명, 일반 단발은 29명이다. 구매 이력 없는 고객은 50명이다.

- 한계:
1. 반복 구매 기준(2건)과 고액 기준(상위 25%)은 업무 기준이 아니라 분포를 보고 임의로 정한 값이므로, 실제 업무에서는 상품 주기나 평균 재구매 간격을 확인해 기준을 정해야 한다.
2. 관측 기간이 13개월이고 주문 건수가 적어서 주문 1건 고객이 앞으로 반복 구매 고객이 될 수 있다.
3. 고객 단위 분석은 반복 구매 이유를 알려 주지 않는다.

### 과제 6. 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸는 프롬프트

고객명 등 개인정보는 넣지 않고, 컬럼 구조와 질문 목록만 제공합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 하고 있습니다. 아래 데이터 구조로는 답할 수 없는 질문을, 현재 데이터로 답할 수 있는 질문으로 바꿔 주세요.

데이터셋(컬럼):
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status(completed/cancelled/refunded), order_month, order_dayofweek
- order_items: order_item_id, order_id, product_id, quantity, unit_price, line_total
- 주문 기간: 2025-09-14 ~ 2026-09-13
- 없는 정보: 할인, 쿠폰, 배송, 원가, 취소·환불 사유, 광고·방문 데이터, 리뷰, 재고

현재 데이터로 답할 수 없는 질문:
1. 취소와 환불이 많았던 이유는 무엇인가?
2. 프로모션을 하면 매출이 얼마나 오르는가?
3. 어떤 카테고리가 이익이 가장 큰가?
4. 어떤 고객이 다시 구매할 의향이 높은가?
5. 인기 있는 카테고리는 무엇인가?

요청:
1. 각 질문이 현재 데이터로 답할 수 없는 이유를 한 문장으로 설명해 주세요.
2. 각 질문을 현재 컬럼만으로 계산할 수 있는 질문으로 바꿔 주세요.
3. 바꾼 질문마다 필요한 데이터셋과 컬럼, 지표, pandas 기능, 분석 단위(주문 / 주문 상세 / 고객)를 정리해 주세요.
4. 바꾼 질문으로 알 수 있는 것과 여전히 알 수 없는 것을 구분해 주세요.
5. 데이터에 없는 정보는 추측하지 말고, 원인에 대한 설명은 가설로 표현해 주세요.
```

#### LLM 답변을 받은 뒤 사람이 확인할 것

- 바꾼 질문이 현재 컬럼(위 목록)만으로 계산 가능한가?
- 없는 컬럼(할인, 원가, 사유 등)을 몰래 가정하지 않았는가?
- "취소 이유", "프로모션 효과" 같은 인과를 단정하지 않고 가설로 표현했는가?
- 분석 단위(주문 / 주문 상세 / 고객)가 질문마다 명확한가?
- 바꾼 질문으로도 여전히 알 수 없는 것이 구분되어 있는가?

## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.


# Chapter 06 답안 양식. 데이터를 보며 질문을 만드는 EDA

> 이 내용을 `chapter06.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름: 배성윤
- GitHub ID: aeuyui
- 작성일: 2026-10-13
- 최종 Notebook URL: https://github.com/aeuyui/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb

## 1. 내가 정한 EDA 질문
| 질문 | 분석 범위 | 지표 | 검증 방법 |
| --- | --- | --- | --- |
| 카테고리별 completed 주문 수량과 금액은 어떻게 다른가? | order_status가 completed인 주문의 주문 상세(order_items)를 products와 product_id로 병합 | 카테고리별 total_quantity, total_sales(line_total 합계), sales_ratio | 카테고리 합계 = 원본 completed line_total 합계, 병합 전후 행 수 동일, 카테고리 누락 0건 |
| completed 기준 월별 매출과 주문 수는 어떻게 변하는가? | completed 주문의 주문 상세를 orders와 order_id로 병합, 주문 기간 2025-09 ~ 2026-09(13개월) | 월별 total_sales, order_count(고유 order_id), avg_order_value | 월별 합계 = 원본 completed line_total 합계, order_date 누락 0건 |
| 상품 수가 많은 카테고리가 매출도 높은가? | products 100개의 카테고리별 상품 수와 completed 카테고리 매출 | 카테고리별 product_count, total_sales, 상품 1개당 매출(total_sales ÷ product_count) | 상품 수 순위와 매출 순위 비교, 카테고리 합계가 원본 completed 합계와 일치하는지 확인 |

### 왜 이 질문을 선택했는가?
세 질문 모두 현재 데이터의 컬럼으로 계산할 수 있다. order_items와 products는 product_id로, order_items와 orders는 order_id로 병합해도 764행이 유지되고 카테고리/주문일 누락이 없었다. 또한 주문의 38.7%(116/300건)가 cancelled 또는 refunded이므로, 금액 분석은 어떤 상태를 포함하는지 먼저 정해야 한다. 그래서 금액성 질문은 모두 completed 범위로 통일했다. 질문 1은 어떤 상품군이 매출에 기여하는지, 질문 2는 시간에 따른 변화, 질문 3은 상품 구성과 매출의 관계를 본다. 세 질문 결과가 서로 같은 총합으로 맞춰지는지 확인할 수 있어서 계산 검증도 쉽다.

## 2. 핵심 집계 결과
- completed 원본 합계: 148,990,000원 (completed 주문 184건, 주문 상세 474행의 line_total 합계) << 합계를 계산하기 위한 별도 계산 셀을 작성하여 실행함.
- 카테고리별 합계: 스포츠 31,743,000원(21.31%), 전자기기 26,400,000원(17.72%), 생활용품 23,915,000원(16.05%), 뷰티 23,383,000원(15.69%), 식품 16,573,000원(11.12%), 도서 16,389,000원(11.00%), 패션 10,587,000원(7.11%). 합계 148,990,000원
- 월별 합계: 13개월 합계 148,990,000원. 최고 2025-12 22,314,000원(24건), 최저 2025-09 2,650,000원(5건)
- 총합 일치 여부: 일치. 원본 completed 합계 = 카테고리 합계 = 월별 합계 = 148,990,000원. 병합 전후 행 수 474행으로 동일, 카테고리 누락 0건, completed 주문 수 184건으로 orders와 같다.

![핵심 EDA 결과](images/step02_eda.png)

### 결과 관찰
- completed 기준 카테고리 매출 1위는 스포츠(21.31%), 7위는 패션(7.11%)이다. 상위 4개 카테고리(스포츠, 전자기기, 생활용품, 뷰티)가 전체의 약 70.8%를 차지한다.
- 식품은 판매 수량(133)이 도서(149)보다 적은데 매출(16,573,000원)은 도서(16,389,000원)보다 크다.
- 월별 매출은 2025-12(22,314,000원)가 가장 높고, 가장 낮은 달은 2025-09(2,650,000원), 그다음이 2026-09(3,348,000원)와 2026-08(4,214,000원)이다.
- 월별 평균 주문 금액은 671,733원(2026-07)부터 1,116,000원(2026-09)까지이다. 2026-09는 주문이 3건뿐이다. completed 전체 평균 주문 금액은 약 809,728원(148,990,000 ÷ 184)이다.
- 전체 상태 기준 합계(255,610,000원)와 비교하면 completed 합계는 58.29%이고 차이는 106,620,000원이다. 뷰티는 전체 상태 기준 2위에서 completed 기준 4위로 내려갔다.
- 상품 수 vs 매출(질문 3): 상품 수 상위 4개 카테고리가 매출 상위 4개와 같고, Spearman 순위 상관은 0.883, 카테고리 합계는 원본 completed 합계와 일치했다(노트북 셀 22 과제 2 출력).

### 나의 해석과 판단
- 취소/환불 주문이 300건 중 116건(약 38.7%)이어서 금액 분석은 completed로 제한했다. 전체 상태로 계산하면 매출이 약 106,620,000원 과대평가되고, 뷰티처럼 취소/환불 비중이 큰 카테고리의 순위도 달라진다.
- 세 합계가 같고 병합 전후 행 수도 같으므로, 병합이나 그룹 집계 과정에서 행이 늘거나 사라지지 않았다고 판단했다.
- 카테고리 순위는 수량과 단가 중 무엇의 영향인지 아직 분리하지 못했으므로, 이 표만으로 선호도나 원인을 말하지 않는다.
- 주문이 3~7건뿐인 달은 평균 주문 금액이 한두 건에 크게 좌우되므로 비교에 신중해야 한다.

### 업무/분석적 의미
같은 범위(completed)로 맞춘 카테고리/월별 표를 7장 시각화와 보고서의 기준표로 쓸 수 있다. 상태 범위에 따라 순위와 최저 월이 달라진다는 점을 확인했으므로, 보고서에는 항상 "completed 기준"이라고 범위를 적어야 한다.

### 한계와 추가 확인 사항
- 매출은 line_total(수량 x 단가) 합계이고, 데이터에 할인/배송비 컬럼이 없어 반영되지 않았다(노트북 셀 6 컬럼 목록).
- 주문 기간이 2025-09-14 ~ 2026-09-13이라 2025-09와 2026-09는 한 달 전체가 아닐 수 있어, 두 달이 최저 월에 포함된 것은 기간 때문일 수 있다.
- refunded 주문을 완전히 제외했으므로 환불 금액 자체의 영향은 이 표로 알 수 없다.
- 2026-08은 주문 16건 중 completed가 5건(약 31%)이라 completed 비율이 낮았는데, 그 이유는 이 데이터만으로 알 수 없다.
- 샘플 데이터이므로 실제 업무 데이터의 분포와 같다고 볼 수 없다.
- 노트북 셀 16~셀 19에서 저장한 reports/ch06_*.csv, ch06_eda_summary.md와 src.eda 결과(셀 18)는 상태를 거르지 않은 전체 상태 기준(합계 255,610,000원)이며, 이 답안의 핵심 집계(completed 기준 148,990,000원)와 범위가 다르다. 보고서에 사용할 때는 범위를 확인해야 한다.

## 3. 관찰 → 가설 → 추가 검증
### 사례 1
- 관찰: 상품 수 상위 4개 카테고리(스포츠 19, 전자기기 17, 생활용품 16, 뷰티 16)가 completed 매출 상위 4개와 같다. 반면 식품은 상품 수가 7개로 가장 적은데 매출은 5위(16,573,000원)이고, 판매 수량(133)이 도서(149)보다 적은데도 매출은 도서(16,389,000원)보다 크다. 상품 1개당 매출은 식품이 약 2,367,571원으로 7개 카테고리 중 가장 높다.
- 가설: 식품은 상품 수는 적지만 평균 단가가 높거나 상품 1개당 판매 수량이 많아서 매출이 클 수 있다. 상품 수가 많은 카테고리의 매출이 높은 것은 상품이 많아 선택지와 상품들의 총 주문 횟수가 많기 때문일 수도 있다. 다만 상품 수 자체의 효과인지는 알 수 없다.
- 추가 검증: 카테고리별 평균 판매 단가(total_sales ÷ total_quantity)와 상품 1개당 판매 수량을 계산하고, 식품 7개 상품별 매출을 확인해 특정 상품에 매출이 몰려 있는지 확인한다.

### 사례 2
- 관찰: 전체 상태 기준 매출 대비 completed 매출의 감소율이 카테고리마다 다르다. 패션 55.5%, 뷰티 50.8%, 식품 50.7%는 크게 줄었고, 생활용품 31.4%, 도서 33.5%, 전자기기 35.6%, 스포츠 36.7%는 상대적으로 덜 줄었다. 그 결과 뷰티는 전체 상태 기준 2위(47,551,000원)에서 completed 기준 4위(23,383,000원)로 내려갔다.
- 가설: 뷰티/패션/식품은 취소/환불로 빠진 주문 상세의 비중이 더 높을 수 있다. 또는 취소/환불된 고액 주문 몇 건이 이 카테고리 매출에 크게 작용했을 수 있다. 또는 표본이 작아 우연일 가능성도 있다.
- 추가 검증: 카테고리별/주문 상태별(completed, cancelled, refunded) 매출과 주문 상세 건수를 계산해 취소/환불 비중을 비교하고, 취소/환불 상위 주문 상세를 확인한다.

### 사례 3
- 관찰: 2026-08은 주문 16건 중 completed가 5건(약 31%)으로 월별 중 가장 낮고, completed 매출 4,214,000원은 같은 달 전체 상태 매출(14,643,000원)의 28.8%이다. 전체 주문의 completed 비율(61.33%)과 차이가 크다. 다음으로 낮은 달은 2026-02(7/16, 약 44%)이다.
- 가설: 이 달에 취소/환불이 늘어난 요인(프로모션, 재고/배송 문제, 특정 결제수단 등)이 있었을 수 있다. 다만 이 데이터에는 원인을 알 수 있는 컬럼이 없다. 또, 주문이 16건뿐이라 비율이 우연히 낮게 나왔을 수도 있다.
- 추가 검증: 월별 order_status 분포(건수/비율)를 계산하고, 2026-08의 cancelled/refunded 주문을 결제수단/카테고리별로 나눠 다른 달과 비교한다. 프로모션이나 재고 정보 같은 외부 자료가 있는지도 확인한다.

## 4. LLM 질문 확장
- Safe Context: 데이터셋 이름과 컬럼 목록(customers에서는 name 제외), 행 수, 주문 기간, completed 기준 카테고리별/월별 집계 요약, 주문 상태 분포만 제공했다. 고객명, customer_id별 구매 내역, 개별 주문 내역은 제공하지 않았다.
- Prompt:

```text
온라인 쇼핑몰 데이터로 EDA를 하고 있습니다. 아래 구조와 집계 요약만 사용해서 다음 분석 질문을 제안해 주세요.

데이터셋(컬럼):
- customers (150행): customer_id, gender, age, city, signup_date
- products (100행): product_id, product_name, category, price
- orders (300행): order_id, customer_id, order_date, payment_method, order_status, order_month, order_dayofweek
- order_items (764행): order_item_id, order_id, product_id, quantity, unit_price, line_total
- 주문 기간: 2025-09-14 ~ 2026-09-13 (13개월)
- 매출 = line_total(수량 x 단가) 합계. 할인, 배송비, 원가 컬럼은 없음.

주문 상태: completed 184건(61.33%), cancelled 64건(21.33%), refunded 52건(17.33%)
completed 주문 기준 합계: 148,990,000원, 주문 184건

completed 기준 카테고리별 요약:
category,product_count,total_quantity,total_sales,sales_ratio
스포츠,19,295,31743000,21.31
전자기기,17,259,26400000,17.72
생활용품,16,272,23915000,16.05
뷰티,16,223,23383000,15.69
식품,7,133,16573000,11.12
도서,14,149,16389000,11.00
패션,11,111,10587000,7.11

completed 기준 월별 요약:
order_month,total_sales,order_count,avg_order_value
2025-09,2650000,5,530000
2025-10,16970000,19,893158
2025-11,11021000,13,847769
2025-12,22314000,24,929750
2026-01,13303000,16,831438
2026-02,5318000,7,759714
2026-03,17529000,24,730375
2026-04,11832000,17,696000
2026-05,16345000,18,908056
2026-06,14070000,18,781667
2026-07,10076000,15,671733
2026-08,4214000,5,842800
2026-09,3348000,3,1116000

관찰한 점:
- 전체 상태 기준 매출 대비 completed 매출이 줄어든 비율은 패션 55.5%, 뷰티 50.8%, 식품 50.7%로 컸고, 생활용품 31.4%, 도서 33.5%, 전자기기 35.6%, 스포츠 36.7%로 작았다.
- 식품은 상품 수가 가장 적은데(7개) 매출은 5위이다.
- 2026-08은 주문 16건 중 completed가 5건뿐이다.

요청:
1. 현재 데이터의 컬럼만으로 계산할 수 있는 다음 EDA 질문 8개를 제안해 주세요.
2. 각 질문마다 필요한 데이터셋과 컬럼, 지표, 사용할 pandas 기능을 정리해 주세요.
3. 현재 데이터로 답할 수 없는 질문 3개를 예로 들고, 각각 답할 수 있는 질문으로 바꿔 주세요.
4. 관찰한 점마다 원인을 단정하지 말고 가설로 표현하고, 가설을 확인할 추가 분석을 제안해 주세요.
5. 데이터에 없는 컬럼이나 정보는 추측하지 마세요.
```

- LLM 제안 요약:

| # | 질문 | 데이터셋 | 컬럼 | 지표 | pandas 기능 | 확인할 점 |
|---|---|---|---|---|---|---|
| 1 | 주문 상태별 비율은 월마다 어떻게 달라지는가? | orders | order_id, order_status, order_month | 월별 전체 주문 수, completed/cancelled/refunded 건수와 비율 | groupby(), size(), unstack(), div(), fillna() | 2026년 8월처럼 completed 비중이 낮은 달을 다른 달과 비교. 원인은 이 분석만으로 알 수 없음 |
| 2 | 카테고리별 취소/환불 비율과 completed 비율은 어떻게 다른가? | orders, order_items, products | order_id, order_status, product_id, category, quantity, line_total | 카테고리별 completed/cancelled/refunded 주문 수와 비율, 상태별 상품 금액 합계 | merge(), groupby(), nunique(), unstack(), crosstab() | 패션/뷰티/식품의 상태별 차이 확인. 한 주문에 여러 카테고리가 있으면 카테고리별 주문 수가 중복 집계될 수 있음 |
| 3 | completed 주문의 월별 상품 금액과 평균 주문 금액은 어떻게 변하는가? | orders, order_items | order_id, order_status, order_month, line_total | 월별 total_sales, 고유 order_count, avg_order_value | merge(), groupby(), agg(), nunique() | 총액 변화가 주문 수와 평균 주문 금액 중 어느 쪽과 함께 나타나는지 확인. 주문별 합계를 먼저 구한 뒤 평균 계산 |
| 4 | 개별 상품별 판매량과 주문 상태 분포에는 어떤 차이가 있는가? | products, order_items, orders | product_id, product_name, category, quantity, line_total, order_id, order_status | 상품별 수량 합계, 상태별 관련 주문 수, completed 상품 금액 | merge(), groupby(), nunique(), sort_values() | 카테고리 합계로 안 보이는 상품별 차이 탐색. 상태 비율의 분모는 해당 상품이 포함된 고유 주문 수 |
| 5 | 결제 수단별 주문 상태 분포는 어떻게 다른가? | orders | order_id, payment_method, order_status | 결제 수단별 주문 수, completed/cancelled/refunded 비율 | crosstab(), groupby(), value_counts(), div() | 차이가 있어도 결제 수단이 취소/환불의 원인이라고 단정할 수 없음 |
| 6 | 고객의 연령대/성별/도시별 구매 패턴은 어떻게 다른가? | customers, orders, order_items | customer_id, age, gender, city, order_id, order_status, line_total | 고객 그룹별 주문 수, 구매 고객 수, completed 주문 금액, 고객당 평균 주문 수 | merge(), groupby(), nunique(), cut(), agg() | 연령대는 분석 편의를 위한 구간으로 정의하고, 그룹별 고객 수를 함께 제시 |
| 7 | 요일별 주문량과 주문 상태 비율은 어떻게 다른가? | orders | order_id, order_dayofweek, order_status | 요일별 주문 수, 상태별 주문 수와 비율 | groupby(), size(), unstack(), reindex() | 특정 요일에 주문이 집중되는지, 상태 비율이 다른지 확인. LLM은 order_dayofweek의 실제 인코딩을 먼저 확인하라고 안내 |
| 8 | 고객별 반복 구매 횟수와 가입일부터 첫 주문까지의 기간은 어떠한가? | customers, orders | customer_id, signup_date, order_id, order_date, order_status | 고객별 전체 주문 수, completed 주문 수, 첫 주문까지 걸린 일수 | merge(), groupby(), nunique(), min(), to_datetime(), 날짜 차이 계산 | 반복 구매 분포와 가입 후 첫 주문까지의 기간 확인. 첫 주문은 전체 기준과 completed 기준으로 구분 가능 |

- 실제 사용 여부: 수정 후 사용
- 판단 이유:
  - 8개 질문 모두 `orders`, `order_items`, `products`, `customers`의 실제 컬럼으로 계산할 수 있고, 할인/원가/취소 사유 같은 없는 컬럼은 쓰지 않았다. 불가능한 질문을 관찰 가능한 질문으로 바꾼 방식도 현재 데이터에 맞다.
  - 질문 1, 2는 3단계의 사례 3, 2를 직접 검증하는 질문이라 사용 가치가 높다. 식품 사례(사례 1)도 LLM이 제안한 상품당 금액/수량 계산으로 확인할 수 있다.
  - 다만 그대로 쓰기 어려운 부분이 있었다. 질문 2와 계산 기준 표에서 카테고리별 비율의 분모가 일관되지 않았고, 질문 3은 이미 2단계에서 계산했으며, 질문 4는 상품당 completed 주문 상세가 평균 약 4.7행이라 상품별 상태 비율의 분모가 작다. 질문 7은 요일이 숫자일 것으로 가정했지만 실제 값은 영문 요일명이었다.

  | 질문 | 판단 |
  |---|---|
  | 1, 2, 5 | 사용 (질문 2는 분모 명시) |
  | 3 | 이미 수행, 중앙값 확인만 추가 |
  | 4 | 수정 후 사용 (상품 단위 상태 비율은 보류, 판매량/금액만) |
  | 6, 7, 8 | 수정 후 사용 |

- 사람이 수정한 내용:
  1. 질문 2의 비율 분모를 "해당 카테고리가 포함된 고유 주문 수"로 명시했다. 한 주문에 여러 카테고리가 들어갈 수 있어 카테고리별 주문 수를 합치면 전체 주문 수보다 커질 수 있으므로, 주문 건수와 주문 상세 행 수를 함께 표시한다.
  2. 질문 3은 이미 같은 표를 만들었으므로 새 분석에서 제외하고, 주문별 금액 합계의 중앙값만 추가 확인 항목으로 남겼다.
  3. 질문 4는 상품별 상태 비율 계산을 보류하고, 상품별 `quantity`와 completed `line_total` 합계, 특히 식품 7개 상품의 매출 분포로 범위를 좁혔다.
  4. 질문 6은 연령대 구간을 직접 정의한다(나이가 19~69세). 그룹별 고객 수를 함께 표시해 도시별 11~21명 같은 작은 그룹이 과대 해석되지 않게 한다.
  5. 질문 7은 `order_dayofweek`가 영문 요일명이므로 숫자 인코딩 확인은 건너뛰고, 월~일 순서로 정렬해 확인한다.
  6. 질문 8은 `signup_date`가 `order_date`보다 늦은 고객이 있는지 먼저 확인한 뒤 첫 주문까지의 기간을 계산한다.
  7. 비율 비교(질문 1, 2, 5, 7)는 월별 주문 수가 6~35건처럼 분모가 작으므로 반드시 건수와 함께 제시한다.

![LLM EDA 질문 확장](images/step04_llm(1).png)

![LLM EDA 질문 확장](images/step04_llm(2).png)

## 5. 다음 분석 우선순위
1. 월별/카테고리별 주문 상태(completed, cancelled, refunded) 비율을 건수와 함께 비교한다. orders의 order_status를 order_month별로 집계하고, order_items/products와 병합해 카테고리별 고유 주문 수와 상태별 line_total을 본다. (LLM 질문 1, 2)
2. 카테고리 매출 차이를 판매 수량과 평균 판매 단가(total_sales / total_quantity)로 나누고, 식품 7개 상품의 매출 분포를 확인한다. 카테고리별 상품당 매출과 판매 수량도 함께 본다. (LLM 질문 4의 수정안)
3. 월별 completed 매출을 주문 수, 평균 주문 금액, 카테고리별 매출로 나눠 어떤 요소와 함께 변하는지 확인한다. 2025-09와 2026-09처럼 일부 날짜만 포함된 달은 완전한 월과 구분한다. (LLM 질문 3의 수정안)

### 가장 먼저 확인하고 싶은 이유
주문 상태 범위가 이후 모든 금액 분석의 기준이기 때문이다. 전체 주문 300건 중 116건(약 38.7%)이 cancelled 또는 refunded이고, completed 합계(148,990,000원)는 전체 상태 합계(255,610,000원)의 58.29%이다. 범위를 바꾼 후에는 뷰티가 2위에서 4위로 내려가고 월별 최저 월도 바뀌었다. 이 분석 하나로 사례 2(카테고리별 감소율 차이)와 사례 3(2026-08의 낮은 completed 비율)을 동시에 검증할 수 있고, 현재 컬럼(order_status, order_month, category)만으로 계산할 수 있다. 상태별 차이를 먼저 알아야 이후 수량/단가 분해와 월별 해석에서 어떤 범위를 쓸지 정할 수 있다.

### 현재 데이터만으로 단정할 수 없는 것
- 취소/환불의 원인: 취소/환불 사유, 배송, 고객 문의 컬럼이 없고 order_status만 있다.
- 월별 매출 변화의 원인: 프로모션, 계절성, 신규 상품, 재고 정보가 없다.
- 카테고리의 선호도/인기: 매출이 높다는 사실만으로 선호도가 높다고 말할 수 없다.
- 순매출/이익: line_total은 수량 x 단가이고, 할인/배송비/원가 컬럼이 없다. 환불 금액이 실제로 어떻게 정산되는지도 알 수 없다.
- 2025-09와 2026-09의 월별 비교: 주문 기간(2025-09-14 ~ 2026-09-13) 때문에 한 달 전체가 아닐 수 있다.
- 2026-08의 낮은 completed 비율이 우연인지: 해당 달 주문이 16건뿐이다.
- 상품 수가 많아서 매출이 높은 것인지: 상품 수와 매출의 관계는 관찰일 뿐, 인과는 확인하지 못했다.
- 샘플 데이터이므로 실제 업무 데이터에도 같은 결과가 나온다고 단정할 수 없다.

업무적으로 확인할 가치가 있는 질문(STEP 8):
1. 2026-08에 취소/환불이 늘 만한 이벤트나 재고/배송 이슈가 있었는가
2. 취소/환불 사유 데이터가 따로 있는가
3. line_total에 할인이 반영되어 있는가
4. 환불 주문의 금액을 매출에서 어떻게 처리하는가

## 최종 체크
- [x] 질문과 지표가 연결되어 있습니다.
- [x] 집계 총합을 검증했습니다.
- [x] 관찰과 가설을 구분했습니다.
- [x] 다음 검증 질문을 작성했습니다.
- [x] LLM 제안을 검증했습니다.
- [x] 최종 Notebook URL을 제출합니다.